In [ ]:
import rasterio as rio
import numpy as np
import geopandas as gpd
from pathlib import Path
import xarray as xr
import rioxarray as rioxr
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import seaborn as sns
from scipy.stats import ks_2samp
import time


### TO DO 7/25
- Look at the difference between Images that Overlap at a higher percentage, and see if there is an edge effect 
- Modify the above figures to only include the full (not the partial and the full)
- plot distribution (not just the mean)
- Try out some kernel density plots of differences between the treatments (not absolute temperature, but temperature difference)
- Send to andrew to forward to Miriam

In [ ]:
# for plotting difference in temp values between overlapping images
def plotDiffImages(pix1, pix2, diff, projstr, outstr1, outstr2):
    fig, ax = plt.subplots()
    pix1.plot.pcolormesh(ax=ax, alpha=0.5, add_colorbar=False)
    pix2.plot.pcolormesh(ax=ax, alpha=0.5, add_colorbar=False)
    diff.plot.pcolormesh(ax=ax, alpha=1, cmap='PiYG')
    ax.axis('equal')
    fig.savefig(f'./figs/ImageDifferenceFigures/{projstr}/TempDiff_{projstr}_{outstr1}_{outstr2}.png', dpi=300)
    plt.close()

def readThermalImg(imgf):
        
    # Load row image
    img_xr = rioxr.open_rasterio(imgf,
                              parse_coordinates=True,
                              masked=True)

    # extract the temp band and convert to celcius
    img_c = (img_xr.sel(band=4)*0.04) - 273.15

    # filter 0 degree C and below as nans
    img_c = img_c.where(img_k > 0)

    return img_c

# Define a function to perform kstest
# NOTE: This isn't really comparing the exact overlap between the 2 images,
# Just the values in the overlap zone...
# For that reason, probably shouldn't trust it!
# 7/21/22
def KSTest_OverlappingPixels(polygon, pix1, pix2):
    
    # Read in both images
    img1 = readThermalImg(imgf1)
    img2 = readThermalImg(imgf2)

    # Clip the pixels of each using the intersection polygon
    pix1 = img1.rio.clip(polygon, drop=True)
    pix2 = img2.rio.clip(polygon, drop=True)
    
    # Flatten data and remove nas
    vals1 = pix1.data.flatten()[np.isnan(pix1.data.flatten(), where=False)]
    vals2 = pix2.data.flatten()[np.isnan(pix1.data.flatten(), where=False)]
    
    if ((vals1.size > 0) & (vals2.size > 0)):
        
        # Run a ks test to compare the values
        results = ks_2samp(vals1, vals2)
    
    else:
        
        results = np.nan
    
    # Return the ks results
    return results

# Function for saving square matrix results
def saveMatrixResults(results, projname, outstr, rowcolnames, m, n):
    # Combine results, and save
    arr = np.array(results)
    # Reshape into a matrix
    arr = arr.reshape(m, n)
    # make into a dataframe, labelling each row,col combination with the image names
    df = pd.DataFrame(arr, index=rowcolnames, columns=rowcolnames)
    # save
    df.to_csv(f'./data/out/{projstr}_{outstr}.csv')
    return df

# Cohen's D - A Measure of effect size 
# aka: a measure of standardized mean difference
# mean1 - mean2 / pooled sample Std
# https://stackoverflow.com/questions/21532471/how-to-calculate-cohens-d-in-python/33002123#33002123
# Other useful links:
# http://ethen8181.github.io/machine-learning/ab_tests/causal_inference/matching.html
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html
# https://en.wikipedia.org/wiki/Effect_size#Difference_family:_Effect_sizes_based_on_differences_between_means
def cohen_d(x,y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1) ** 2 + (ny-1)*np.std(y, ddof=1) ** 2) / dof)

In [1]:
def meanDiff_ImagesandExclosures(pix1_overlap, pix2_overlap, i1, i2, shp=shp, printvalues=False, includepartial=False):
    
    # This Function clips the overlap zones of 2 images with exclosure polygons
    # It returns:
    # the pixel values (OPdiff_img1, OPdiff_img2)
    # The mean differences between images in the overlap zone (imgdiff) 
    # The mean differences between exclosure treatments (df_img12)
    # NOTE: This may not work well if the area overlaps more than 2 exclosure treatments (Open & Partial & Full all at once)
    
    # Inputs pix1_overlap and pix2_overlap are the overlapping pixels of 2 image files
    # i1 and i2 are the full paths to both img files
    # shp is a geodataframe of the shapefile polygons
    # print values will spit out a mean absolute difference if True
    
    # 7/25 - Filter shapefile for only the full and open exclosure types
    # Andrew doesn't want partial in the equation
    if includepartial:
        shp_filter = shp.copy()
    else:
        shp_filter = shp[shp.Exclosure != 'Partial']
    
    # Initialize df list for loop
    df_list = []

    # For each polygon treatment in the shapefile
    for ID, g in zip(shp_filter.index, shp_filter.geometry):

        # For each image with overlapping pixels
        for pix, i in zip([pix1_overlap, pix2_overlap], [i1, i2]):

            # Try clipping the image with the feature in the shapefile
            try:
                
                p = pix.rio.clip([g], shp.crs, drop=True)

                # Initialize an empty df
                df = pd.DataFrame()

                # img name
                iname = Path(i).name

                # Make an exploded dataframe, with each pixel noted by it's treatment and imgfile
                df = pd.DataFrame({'Temperature':p.data.flatten(),
                                   'Exclosure':shp_filter.Exclosure.iloc[ID],
                                   'imgf':iname})

                # Drop na rows
                df.dropna(axis=0, how='any', inplace=True)

                # store DataFrame in list
                df_list.append(df.copy(deep=True))

            # if it fails, no pixels, move on
            except:
                continue

    # Concat all the dfs to make a full df 
    df_img12 = pd.concat(df_list, ignore_index=True, sort=False)

    # set nodata values again (rioxarray changes it to 3.4028234663852886e+38)
    df_img12[df_img12 == 3.4028234663852886e+38] = np.nan

    # Group by Exclosure and Images
    df_img12_g = df_img12.groupby(['Exclosure','imgf'])
    df_img12_img_g = df_img12.groupby(['imgf'])
    
    # If there are 2 images and 2 exclosure types to work with:
    if (df_img12_g.mean().shape[0] >= 4):
    
        # Take the absolute mean difference in exclosures per image
        OPdiff_img1 = df_img12_g.mean().iloc[0] - df_img12_g.mean().iloc[2]
        OPdiff_img2 = df_img12_g.mean().iloc[1] - df_img12_g.mean().iloc[3]

        # Just return single value
        OPdiff_img1 = OPdiff_img1.values[0]
        OPdiff_img2 = OPdiff_img2.values[0]

        # Mean difference between images
        imgdiff = df_img12_img_g.mean().iloc[0] - df_img12_img_g.mean().iloc[1]
        imgdiff = imgdiff.values[0]
    
    else:
        
        OPdiff_img1 = np.nan
        OPdiff_img2 = np.nan
        imgdiff = np.nan
        
    # Print the means
    # What we want to see here is that the differences in exclosure temperatures
    # are consistent between images    
    if printvalues:
        print(f'Overlap Zones \n Exclosure Temp Difference: \n \t {Path(i1).name}: {OPdiff_img1} \n \t {Path(i2).name}: {OPdiff_img2}')
        print(f' Difference Between Images: \n \t {imgdiff}')
    
    # Return df of pixels, and mean difference values
    return OPdiff_img1, OPdiff_img2, imgdiff, df_img12

NameError: name 'shp' is not defined

In [ ]:
# Function for computing difference in overlapping pixels
# Uses repojection and matching to match img1 to img2
# then computes difference between pixels
# See: https://corteva.github.io/rioxarray/stable/examples/reproject_match.html
def diff_OverlappingPixels(polygon, imgf1, imgf2):
    
    # Make outstrings for plotting from the filenames
    imgname1 = Path(imgf1).name.split('.tif')[0]
    imgname2 = Path(imgf2).name.split('.tif')[0]

    # If they're the same image, just skip it!
    if imgname1==imgname2:
        
        diff = np.nan
        diff_mean = 0
        diff_std = 0
        cd = 0
        OPdiff_img1 = np.nan
        OPdiff_img2 = np.nan
        imgdiff = 0
        df_img12 = np.nan
        
    # Otherwise...
    else:
        
        # Read in both images
        # Returns an image in Celcius
        img1 = readThermalImg(imgf1)
        img2 = readThermalImg(imgf2)
        
        # Clip the pixels of each using the intersection polygon
        pix1 = img1.rio.clip([polygon], drop=True)
        pix2 = img2.rio.clip([polygon], drop=True)
        
        # set no data
        pix1 = pix1.where((pix1 < 150) & (pix1 >= 0))
        pix2 = pix2.where((pix2 < 150) & (pix2 >= 0))

        # Make a new pix2 matching the projection of pix1
        pix2_match = pix2.rio.reproject_match(pix1)

        # Set the coordinates of the grid (same as pix1)
        pix2_match.assign_coords({"x": pix1.x,
                                  "y": pix1.y})

        # Compute the pixel-wise difference
        diff = pix1 - pix2_match

        # Compute Summary Stats (absolute values)
        diff_mean = np.nanmean(np.abs(diff.data.flatten()))
        diff_std = np.nanstd(np.abs(diff.data.flatten()))
                                   
        # Plot the differences, if there are any
        if not(np.isnan(diff_mean)):
            
            plotDiffImages(pix1=pix1,
                           pix2=pix2_match,
                           diff=diff,
                           projstr=projstr,
                           outstr1=imgname1,
                           outstr2=imgname2)
         
        # Test Exclosure differences within overlap areas
        
        # Get only the overlap pixels in each image
        # Cutting out edges that are in the overlap zones, but are na
        pix1_overlap = pix1.where(diff.notnull())
        pix2_overlap = pix2_match.where(diff.notnull())
        
        # Compute cohen's d (measure of effect size)
        # Standardized mean difference between the two
        cd = cohen_d(pix1_overlap, pix2_overlap)
        
        # Apply a function to clip the overlapping areas with polygons again
        # and return a variety of values
        OPdiff_img1, OPdiff_img2, imgdiff, df_img12 = meanDiff_ImagesandExclosures(pix1_overlap,
                                                                                   pix2_overlap,
                                                                                   i1=imgf1,
                                                                                   i2=imgf2,
                                                                                   printvalues=False)
    
    return diff, diff_mean, diff_std, cd, OPdiff_img1, OPdiff_img2, imgdiff, df_img12

In [ ]:
# Function for Making Overlapping Image
# Time and Space Boxplots
# Pipeline for getting difference in Time and Space
# 7/25/22 PB
# Need to Incorporate this into the main loop 
# and run 

# NOTE: NEEDS WORK especially the first part!!!
# This is a tweaked version of the above
# 

def timevsspaceBoxplot(df_img12, i1, i2, projstr='Letaba'):
    
    # Step 1 - create new version of df_img12
    # NEED TO FIX INPUTS 
    # AND INCORPORATE INTO LOOP
    # 7/25/2022
    
    # Initialize df list for loop
    df_list = []
    p_list = []

    # For each polygon treatment in the shapefile
    for ID, g in zip(shp.index, shp.geometry):

            # For each image with overlapping pixels
            # for pix, i in zip([pix1_overlap, pix2_overlap], [i1, i2]):

            # Try clipping both images with the features in the shapefile
            try:

                # clip xarray objects of images
                p1 = pix1_overlap.rio.clip([g], shp.crs, drop=True)
                p2 = pix2_overlap.rio.clip([g], shp.crs, drop=True)

                # Make a df from each
                pix1_overlap_df = p1.to_dataframe(name=Path(i1).name)
                pix2_overlap_df = p2.to_dataframe(name=Path(i2).name)

                # Drop na rows
                pix1_overlap_df.dropna(axis=0, inplace=True)
                pix2_overlap_df.dropna(axis=0, inplace=True)

                # Drop cols
                pix1_overlap_df.drop(['band', 'spatial_ref'], axis=1, inplace=True)
                pix2_overlap_df.drop(['band', 'spatial_ref'], axis=1, inplace=True)

                # Merge the dataframes (merging on the x, y)
                # This makes a dataframe with x, y, img1 temp, img2 temp, exclosure
                # 1 row for each pixel
                pix12_overlap_df = pd.merge(pix1_overlap_df, pix2_overlap_df, on=['y', 'x'])

                # Assign a new column with the Exclosure
                pix12_overlap_df = pix12_overlap_df.assign(Exclosure=shp.Exclosure[ID])

                # store DataFrame in list
                df_list.append(pix12_overlap_df.copy(deep=True))

            # if it fails, no pixels, move on
            except:
                continue

    # Concat all the dfs to make a full df 
    df_img12 = pd.concat(df_list, ignore_index=True, sort=False)

    # set nodata values again (rioxarray changes it to 3.4028234663852886e+38)
    df_img12[df_img12 == 3.4028234663852886e+38] = np.nan
    
    # STEP 2 - Get differences inside and out

    meanDiff_byImage_outside = []
    meanDiff_byImage_inside = []
    mdE_df_list = []

    # Group each pixel by exclosure
    df_img12_g = df_img12.groupby(by='Exclosure')

    # Now make a bunch of samples for bootstrapping
    for i in range(0, 100):

        i1name = Path(i1).name
        i2name = Path(i2).name

        # Sample df, keeping 25% of the rows in open and partial exclosures
        df_img12_samp = df_img12_g.sample(frac=0.25)

        # Group by Exclosure again, and ...
        df_img12_samp_open = df_img12_samp.loc[df_img12_samp['Exclosure'] == 'Open']
        df_img12_samp_inside = df_img12_samp.loc[((df_img12_samp['Exclosure'] == 'Partial') & (df_img12_samp['Exclosure'] == 'Full'))]

        # 1) Compute the mean difference between exclosures (Space)
        # outside - inside means
        # Note: This will give 2 values, one mean diff per image
        mdE = df_img12_samp_open.drop('Exclosure', axis=1).mean() - df_img12_samp_inside.drop('Exclosure', axis=1).mean()

        # save each mean difference
        # meanDiff_byExclosure_Image1.append(mdE[0])
        # meanDiff_byExclosure_Image2.append(mdE[1])
        # save in a df
        mdE_df = pd.DataFrame(mdE, columns=['ExclosureDiff']).reset_index()
        mdE_df.rename(columns={"index": "imgf"}, inplace=True)

        # 2)  Compute the mean difference between images (Time)

        # All overlapping values difference
        mdI = np.mean(df_img12_samp[i1name] - df_img12_samp[i2name])

        # Overlapping within the open exclosure difference
        mdI_outside = np.mean(df_img12_samp_open[i1name] - df_img12_samp_open[i2name])

        # Overlapping inside the exclosure difference
        mdI_inside = np.mean(df_img12_samp_inside[i1name] - df_img12_samp_inside[i2name])

        # Make a new df with all 3 
        # meanDiff_Time_df = pd.DataFrame({'Overall':[], 'Inside':[], 'Outside':[]})

        meanDiff_byImage.append(mdI)
        meanDiff_byImage_outside.append(mdI_outside)
        meanDiff_byImage_inside.append(mdI_inside)

        # Add the mean difference to the meanDiff by exclosure df
        mdE_df = mdE_df.assign(TimeDiff = mdI)

        # Append to the list to concat outside of the loop
        mdE_df_list.append(mdE_df)

    # Concat
    # each row is now an iteration with the current imagefile
    meanDiff_byEandI_df = pd.concat(mdE_df_list, ignore_index=True)
    meanDiff_byEandI_df.head()

    # Make figure output
    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True)
    sns.boxplot(data=meanDiff_byEandI_df, ax=ax1,
                x='imgf', y='ExclosureDiff')
    sns.boxplot(data=meanDiff_byEandI_df.loc[meanDiff_byEandI_df['imgf'] == i1name],
                ax=ax2, x='imgf', y='TimeDiff', color='g')
    ax1.set_ylabel('Bootstrapped Mean Temperature Difference [C]')
    ax2.set(xticklabels=[])
    ax2.set(xlabel=f'Bootstrapped Difference between Images \n {i1name} - {i2name}')
    fig.legend()
    ax1.set_title('Exclosure Difference')
    ax2.set_ylabel('')
    fig.savefig(f'./figs/TimevsSpace/{projstr}/Boxplots_TimevsSpace_{i1name}_{i2name}.png', dpi=300)

In [ ]:
# Import functions
# from Functions_ThermalLandscapes import KSTest_OverlappingPixels, readThermalImg, saveMatrixResults

# Make a geoseries of all the img_polygons
poly_gs = gpd.GeoSeries(data=img_dict['boundary_poly'], 
                        index=img_dict['imgf'],
                        crs='EPSG:32736')

diff_list = []
diffmean_list = []
diffstd_list = []
img1_list = []
img2_list = []
intersects_list = []
cd_overlap_list = []
odiff_img1_mean_list = []
odiff_img2_mean_list = []
odiff_imgs_list = []
odf_imgs_list = []


for img1, poly1 in poly_gs.iteritems():
    for img2, poly2 in poly_gs.iteritems():
        
        start = time.time()
        
        img1_list.append(img1)
        img2_list.append(img2)
        
        intersects_list.append(poly1.intersects(poly2))
        
        # Check for intersection
        if poly1.intersects(poly2):

            # Get the intersection
            intersection = poly1.intersection(poly2)
            
            # Apply functions to compare the overlapping pixels
            # Pixel Differences
            d, d_mean, d_std, cd, odiff_img1_mean, odiff_img2_mean, odiff_imgs, odf_imgs =  diff_OverlappingPixels(intersection,
                                                                                                        f'{img_dir}/{img1}',
                                                                                                        f'{img_dir}/{img2}')
            diff_list.append(d)
            diffmean_list.append(d_mean)
            diffstd_list.append(d_std)
            cd_overlap_list.append(cd)
            odiff_img1_mean_list.append(odiff_img1_mean)
            odiff_img2_mean_list.append(odiff_img2_mean)
            odiff_imgs_list.append(odiff_imgs)
            odf_imgs_list.append(odf_imgs)
            
        else:
            
            diff_list.append(np.nan)
            diffmean_list.append(np.nan)
            diffstd_list.append(np.nan)
            cd_overlap_list.append(np.nan)
            odiff_img1_mean_list.append(np.nan)
            odiff_img2_mean_list.append(np.nan)
            odiff_imgs_list.append(np.nan)
            odf_imgs_list.append(np.nan)
            
        end = time.time()
        
        # print(f'{end-start} seconds')

diffmean_df = saveMatrixResults(diffmean_list, 
                                projname=projstr,
                                outstr='MeanPixelDiff_OverlapZones',
                                rowcolnames=poly_gs.index,
                                m=poly_gs.shape[0],
                                n=poly_gs.shape[0])

diffstd_df = saveMatrixResults(diffstd_list, 
                               projname=projstr,
                               outstr='StdPixelDiff_OverlapZones',
                               rowcolnames=poly_gs.index,
                               m=poly_gs.shape[0],
                               n=poly_gs.shape[0])

cd_overlap_df = saveMatrixResults(cd_overlap_list, 
                                  projname=projstr,
                                  outstr='CohensD_OverlapZones',
                                  rowcolnames=poly_gs.index,
                                  m=poly_gs.shape[0],
                                  n=poly_gs.shape[0])

odiff_img1_df = saveMatrixResults(odiff_img1_mean_list, 
                                  projname=projstr,
                                  outstr='ExclosureDiffsImg1_OverlapZones',
                                  rowcolnames=poly_gs.index,
                                  m=poly_gs.shape[0],
                                  n=poly_gs.shape[0])

odiff_img2_df = saveMatrixResults(odiff_img2_mean_list, 
                                  projname=projstr,
                                  outstr='ExclosureDiffsImg2_OverlapZones',
                                  rowcolnames=poly_gs.index,
                                  m=poly_gs.shape[0],
                                  n=poly_gs.shape[0])

odiff_imgs_df = saveMatrixResults(odiff_imgs_list, 
                                  projname=projstr,
                                  outstr='ImageDiffs_OverlapZones',
                                  rowcolnames=poly_gs.index,
                                  m=poly_gs.shape[0],
                                  n=poly_gs.shape[0])

In [ ]:
# Plot Some stuff

# Make a new df
plotdf = pd.DataFrame({'img1':img1_list,
                       'img2':img2_list,
                       'OverlapDiff':odiff_imgs_list,
                       'ExclosureDiff_img1':odiff_img1_mean_list,
                       'ExclosureDiff_img2':odiff_img2_mean_list})

plotdf.dropna(inplace=True)
plotdf.head()

fig, (ax1, ax2) = plt.subplots(2)
plotdf.plot.scatter(x='img1', y='OverlapDiff', c='b', s=50, ax=ax1, label='Time Difference')
plotdf.plot.scatter(x='img1', y='ExclosureDiff_img1', c='r', s=50, ax=ax1, label='Treatment Difference')
# ax1.set_xticks(ax1.get_xticks())
# ax1.set_xticklabels(ax1.get_xticklabels(), rotation=90, ha='right')
plotdf.plot.scatter(x='img2', y='OverlapDiff', c='b', s=50, ax=ax2, label='Time Difference')
plotdf.plot.scatter(x='img2', y='ExclosureDiff_img2', c='r', s=50,  ax=ax2, label='Treatment Difference')
ax1.legend()
ax1.set_ylabel('Difference in Temp [C]')
ax2.set_ylabel('Difference in Temp [C]')
plt.xticks(rotation=90, ha='right')

In [ ]:
## # Import functions
# from Functions_ThermalLandscapes import KSTest_OverlappingPixels, readThermalImg, saveMatrixResults

# Make a geoseries of all the img_polygons
poly_gs = gpd.GeoSeries(data=img_dict['boundary_poly'], 
                        index=img_dict['imgf'],
                        crs='EPSG:32736')

# ks_results = []
diff_list = []
diffmean_list = []
diffstd_list = []
img1_list = []
img2_list = []
intersects_list = []

for img1, poly1 in poly_gs.iteritems():
    for img2, poly2 in poly_gs.iteritems():
        
        strt = time.time()
        
        img1_list.append(img1)
        img2_list.append(img2)
        
        intersects_list.append(poly1.intersects(poly2))
        
        # Check for intersection
        if poly1.intersects(poly2):

            # Get the intersection
            intersection = poly1.intersection(poly2)
            
            
            # apply functions to compare the overlapping pixels
            
            # # KS tests
            # Commented out, not running anymore
            # ks = KSTest_OverlappingPixels([intersection],
            #                               f'{img_dir}/{img1}',
            #                               f'{img_dir}/{img2}')
            # # Unpack each item (they are listen as kstest result, can just save the pvalue
            # p = np.round(ks.pvalue, decimals=4)
            # ks_results.append(p)
            
            # Pixel Differences
            d, d_mean, d_std =  diff_OverlappingPixels(intersection,
                                                       f'{img_dir}/{img1}',
                                                       f'{img_dir}/{img2}')
            
            
            diff_list.append(d)
            diffmean_list.append(d_mean)
            diffstd_list.append(d_std)
            
        else:
            
            # ks_results.append(np.nan)
            diff_list.append(np.nan)
            diffmean_list.append(np.nan)
            diffstd_list.append(np.nan)
            
        end = time.time()
        
        # print(f'{end-strt} seconds')
        
# Combine results, and save
# Commented out - not running anymore
# ks_df = saveMatrixResults(ks_results, 
#                           projname=projstr,
#                           outstr='KSTest_OverlapZones',
#                           rowcolnames=poly_gs.index,
#                           m=poly_gs.shape[0],
#                           n=poly_gs.shape[0])

diffmean_df = saveMatrixResults(diffmean_list, 
                          projname=projstr,
                          outstr='MeanPixelDiff_OverlapZones',
                          rowcolnames=poly_gs.index,
                          m=poly_gs.shape[0],
                          n=poly_gs.shape[0])

diffstd_df = saveMatrixResults(diffstd_list, 
                          projname=projstr,
                          outstr='StdPixelDiff_OverlapZones',
                          rowcolnames=poly_gs.index,
                          m=poly_gs.shape[0],
                          n=poly_gs.shape[0])

# ks_arr = np.array(ks_results)
# # Reshape into a matrix
# ks_arr = ks_arr.reshape(poly_gs.shape[0], poly_gs.shape[0])
# # make into a dataframe, labelling each row,col combination with the image names
# ks_df = pd.DataFrame(ks_arr, index=poly_gs.index, columns=poly_gs.index)
# #save
# ks_df.to_csv(f'./data/out/{projstr}_KSTest_OverlapZones.csv')TESTING

In [ ]:
# Import functions
# from Functions_ThermalLandscapes import KSTest_OverlappingPixels, readThermalImg, saveMatrixResults

# Make a geoseries of all the img_polygons
poly_gs = gpd.GeoSeries(data=img_dict['boundary_poly'], 
                        index=img_dict['imgf'],
                        crs='EPSG:32736')

# ks_results = []
diff_list = []
diffmean_list = []
diffstd_list = []
img1_list = []
img2_list = []
intersects_list = []

for img1, poly1 in poly_gs.iteritems():
    for img2, poly2 in poly_gs.iteritems():
        
        strt = time.time()
        
        img1_list.append(img1)
        img2_list.append(img2)
        
        intersects_list.append(poly1.intersects(poly2))
        
        # Check for intersection
        if poly1.intersects(poly2):

            # Get the intersection
            intersection = poly1.intersection(poly2)
            
            
            # apply functions to compare the overlapping pixels
            
            # # KS tests
            # Commented out, not running anymore
            # ks = KSTest_OverlappingPixels([intersection],
            #                               f'{img_dir}/{img1}',
            #                               f'{img_dir}/{img2}')
            # # Unpack each item (they are listen as kstest result, can just save the pvalue
            # p = np.round(ks.pvalue, decimals=4)
            # ks_results.append(p)
            
            # Pixel Differences
            d, d_mean, d_std =  diff_OverlappingPixels(intersection,
                                                       f'{img_dir}/{img1}',
                                                       f'{img_dir}/{img2}')
            
            
            diff_list.append(d)
            diffmean_list.append(d_mean)
            diffstd_list.append(d_std)
            
        else:
            
            # ks_results.append(np.nan)
            diff_list.append(np.nan)
            diffmean_list.append(np.nan)
            diffstd_list.append(np.nan)
            
        end = time.time()
        
        # print(f'{end-strt} seconds')
        
# Combine results, and save
# Commented out - not running anymore
# ks_df = saveMatrixResults(ks_results, 
#                           projname=projstr,
#                           outstr='KSTest_OverlapZones',
#                           rowcolnames=poly_gs.index,
#                           m=poly_gs.shape[0],
#                           n=poly_gs.shape[0])

diffmean_df = saveMatrixResults(diffmean_list, 
                          projname=projstr,
                          outstr='MeanPixelDiff_OverlapZones',
                          rowcolnames=poly_gs.index,
                          m=poly_gs.shape[0],
                          n=poly_gs.shape[0])

diffstd_df = saveMatrixResults(diffstd_list, 
                          projname=projstr,
                          outstr='StdPixelDiff_OverlapZones',
                          rowcolnames=poly_gs.index,
                          m=poly_gs.shape[0],
                          n=poly_gs.shape[0])

# ks_arr = np.array(ks_results)
# # Reshape into a matrix
# ks_arr = ks_arr.reshape(poly_gs.shape[0], poly_gs.shape[0])
# # make into a dataframe, labelling each row,col combination with the image names
# ks_df = pd.DataFrame(ks_arr, index=poly_gs.index, columns=poly_gs.index)
# #save
# ks_df.to_csv(f'./data/out/{projstr}_KSTest_OverlapZones.csv')

In [ ]:
# Beginning of Steps for Space vs Time Analysis
# 7/25/22

# i1 = f'{img_dir}/cam1-001-002044-0.tif'
# i2 = f'{img_dir}/cam1-001-002009-0.tif'
i1 = f'{img_dir}/cam1-001-001014-0.tif'
i2 = f'{img_dir}/cam1-001-000602-0.tif'

# Read in both images
img1 = readThermalImg(i1)
img2 = readThermalImg(i2)

# Get the intersection of the 2 polygons
poly1 = poly_gs.loc[poly_gs.index == 'cam1-001-001014-0.tif']
poly2 =  poly_gs.loc[poly_gs.index == 'cam1-001-000602-0.tif']

# Get the intersection
polygon = poly1.intersection(poly2[0])

# Clip the pixels of each using the intersection polygon
pix1 = img1.rio.clip(polygon, drop=True)
pix2 = img2.rio.clip(polygon, drop=True)

# # Select temp band
# pix1 = pix1.sel(band=4)*0.04 - 273.15
# pix2 = pix2.sel(band=4)*0.04 - 273.15

# set no data
pix1 = pix1.where((pix1 < 150) & (pix1 >= 0))
pix2 = pix2.where((pix2 < 150) & (pix2 >= 0))

# Transpose dimensions (just for the below function)
# pix1 = pix1.transpose('band', 'y', 'x')
# pix2 = pix2.transpose('band', 'y', 'x')

# Make a new pix2 matching the projection of pix1
pix2_match = pix2.rio.reproject_match(pix1)

# Set the coordinates of the grid (same as pix1)
pix2_match.assign_coords({"x": pix1.x,
                          "y": pix1.y})

# compute diff
diff = pix1 - pix2_match

diff_mean = np.nanmean(np.abs(diff.data.flatten()))
diff_std = np.nanstd(np.abs(diff.data.flatten()))

In [ ]:
# Get only the overlap pixels in each image
# Cutting out edges that are in the overlap zones, but are na
# where the difference thing is na 
pix1_overlap = pix1.where(diff.notnull())
pix2_overlap = pix2_match.where(diff.notnull())

# Calculate Cohen's D for overlapping areas of images
cd = cohen_d(pix1_overlap, pix2_overlap)

In [ ]:
OPdiff_img1, OPdiff_img2, imgdiff, df_img12 = meanDiff_ImagesandExclosures(pix1_overlap, pix2_overlap, i1, i2, printvalues=True)

In [ ]:
# Clip the overlaps of each image with the shapefile features
# Get the mean difference between images in the overlap zone

# Initialize df list for loop
df_list = []
p_list = []

# For each polygon treatment in the shapefile
for ID, g in zip(shp.index, shp.geometry):
    
    # For each image with overlapping pixels
    for pix, i in zip([pix1_overlap, pix2_overlap], [i1, i2]):

        # Try clipping the image with the feature in the shapefile
        try:
            
            p = pix.rio.clip([g], shp.crs, drop=True)
            
            p_list.append(p)

            # Initialize an empty df
            df = pd.DataFrame()
            
            # img name
            iname = Path(i).name

            # Make an exploded dataframe, with each pixel noted by it's treatment, topo, and imgfile
            df = pd.DataFrame({'Temperature':p.data.flatten(),
                               'Exclosure':shp.Exclosure.iloc[ID],
                               'imgf':iname})

            # Drop na rows
            df.dropna(axis=0, how='any', inplace=True)

            # store DataFrame in list
            df_list.append(df.copy(deep=True))

        # if it fails, no pixels, move on
        except:
            continue
        
# Concat all the dfs to make a full df 
df_img12 = pd.concat(df_list, ignore_index=True, sort=False)

# set nodata values again (rioxarray changes it to 3.4028234663852886e+38)
df_img12[df_img12 == 3.4028234663852886e+38] = np.nan

# Group by Exclosure and Images
df_img12_g = df_img12.groupby(['Exclosure','imgf'])
df_img12_img_g = df_img12.groupby(['imgf'])

# Print the means
# What we want to see here is that the differences in exclosure temperatures
# are consistent between images
# Take the absolute mean difference in exclosures per image
OPdiff_img1 = df_img12_g.mean().iloc[0] - df_img12_g.mean().iloc[2]
OPdiff_img2 = df_img12_g.mean().iloc[1] - df_img12_g.mean().iloc[3] 

# Mean difference between images
imgdiff = df_img12_img_g.mean().iloc[0] - df_img12_img_g.mean().iloc[1]

print(f'Overlap Zones \n Exclosure Temp Difference: \n \t {Path(i1).name}: {OPdiff_img1.values[0]} \n \t {Path(i2).name}: {OPdiff_img2.values[0]}')
print(f' Difference Between Images: \n \t {imgdiff.values[0]}')

In [ ]:
# Initialize df list for loop
df_list = []
p_list = []

# For each polygon treatment in the shapefile
for ID, g in zip(shp.index, shp.geometry):
    
        # For each image with overlapping pixels
        # for pix, i in zip([pix1_overlap, pix2_overlap], [i1, i2]):

        # Try clipping both images with the features in the shapefile
        try:

            # clip xarray objects of images
            p1 = pix1_overlap.rio.clip([g], shp.crs, drop=True)
            p2 = pix2_overlap.rio.clip([g], shp.crs, drop=True)
            
            # Make a df from each
            pix1_overlap_df = p1.to_dataframe(name=Path(i1).name)
            pix2_overlap_df = p2.to_dataframe(name=Path(i2).name)

            # Drop na rows
            pix1_overlap_df.dropna(axis=0, inplace=True)
            pix2_overlap_df.dropna(axis=0, inplace=True)

            # Drop cols
            pix1_overlap_df.drop(['band', 'spatial_ref'], axis=1, inplace=True)
            pix2_overlap_df.drop(['band', 'spatial_ref'], axis=1, inplace=True)
            
            # Merge the dataframes (merging on the x, y)
            # This makes a dataframe with x, y, img1 temp, img2 temp, exclosure
            # 1 row for each pixel
            pix12_overlap_df = pd.merge(pix1_overlap_df, pix2_overlap_df, on=['y', 'x'])
            
            # Assign a new column with the Exclosure
            pix12_overlap_df = pix12_overlap_df.assign(Exclosure=shp.Exclosure[ID])

            # store DataFrame in list
            df_list.append(pix12_overlap_df.copy(deep=True))

        # if it fails, no pixels, move on
        except:
            continue
        
# Concat all the dfs to make a full df 
df_img12 = pd.concat(df_list, ignore_index=True, sort=False)

# set nodata values again (rioxarray changes it to 3.4028234663852886e+38)
df_img12[df_img12 == 3.4028234663852886e+38] = np.nan

In [ ]:
# Pipeline for getting difference in Time and Space
# 7/25/22 PB
# Need to Incorporate this into the main loop 
# and run 

meanDiff_byImage_outside = []
meanDiff_byImage_inside = []
mdE_df_list = []

# Group each pixel by exclosure
df_img12_g = df_img12.groupby(by='Exclosure')

# Now make a bunch of samples for bootstrapping
for i in range(0, 100):
    
    # Sample df, keeping 25% of the rows in open and partial exclosures
    df_img12_samp = df_img12_g.sample(frac=0.25)

    # Group by Exclosure again, and ...
    df_img12_samp_open = df_img12_samp.loc[df_img12_samp['Exclosure'] == 'Open']
    df_img12_samp_inside = df_img12_samp.loc[df_img12_samp['Exclosure'] == 'Partial']
    
    # 1) Compute the mean difference between exclosures (Space)
    # outside - inside means
    # Note: This will give 2 values, one mean diff per image
    mdE = df_img12_samp_open.drop('Exclosure', axis=1).mean() - df_img12_samp_inside.drop('Exclosure', axis=1).mean()
    
    # save each mean difference
    # meanDiff_byExclosure_Image1.append(mdE[0])
    # meanDiff_byExclosure_Image2.append(mdE[1])
    # save in a df
    mdE_df = pd.DataFrame(mdE, columns=['ExclosureDiff']).reset_index()
    mdE_df.rename(columns={"index": "imgf"}, inplace=True)
    
    # 2)  Compute the mean difference between images (Time)
    
    # All overlapping values difference
    mdI = np.mean(df_img12_samp[Path(i1).name] - df_img12_samp[Path(i2).name])
    
    # Overlapping within the open exclosure difference
    mdI_outside = np.mean(df_img12_samp_open[Path(i1).name] - df_img12_samp_open[Path(i2).name])
    
    # Overlapping inside the exclosure difference
    mdI_inside = np.mean(df_img12_samp_inside[Path(i1).name] - df_img12_samp_inside[Path(i2).name])
    
    # Make a new df with all 3 
    # meanDiff_Time_df = pd.DataFrame({'Overall':[], 'Inside':[], 'Outside':[]})
    
    meanDiff_byImage.append(mdI)
    meanDiff_byImage_outside.append(mdI_outside)
    meanDiff_byImage_inside.append(mdI_inside)
    
    # Add the mean difference to the meanDiff by exclosure df
    mdE_df = mdE_df.assign(TimeDiff = mdI)
        
    # Append to the list to concat outside of the loop
    mdE_df_list.append(mdE_df)

# Concat
# each row is now an iteration with the current imagefile
meanDiff_byEandI_df = pd.concat(mdE_df_list, ignore_index=True)
meanDiff_byEandI_df.head()

fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True)
sns.boxplot(data=meanDiff_byEandI_df, ax=ax1,
            x='imgf', y='ExclosureDiff')
sns.boxplot(data=meanDiff_byEandI_df.loc[meanDiff_byEandI_df['imgf'] == Path(i2).name],
            ax=ax2, x='imgf', y='TimeDiff', color='g')
ax1.set_ylabel('Temperature Difference [C]')
ax2.set(xticklabels=[])
ax2.set(xlabel=f'Difference between Images \n {Path(i1).name} - {Path(i2).name}')
fig.legend()
ax1.set_title('Exclosure Difference')
ax2.set_ylabel('')

In [ ]:
# Concat
# each row is now an iteration with the current imagefile
meanDiff_byEandI_df = pd.concat(mdE_df_list, ignore_index=True)
meanDiff_byEandI_df.head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True)
sns.boxplot(data=meanDiff_byEandI_df, ax=ax1,
            x='imgf', y='ExclosureDiff')
sns.boxplot(data=meanDiff_byEandI_df.loc[meanDiff_byEandI_df['imgf'] == Path(i2).name],
            ax=ax2, x='imgf', y='TimeDiff', color='g')
ax1.set_ylabel('Temperature Difference [C]')
ax2.set(xticklabels=[])
ax2.set(xlabel=f'Difference between Images \n {Path(i1).name} - {Path(i2).name}')
fig.legend()
ax1.set_title('Treatment')
ax2.set_title('Time')
ax2.set_ylabel('')

In [ ]:
fig, ax = plt.subplots()
sns.kdeplot(data=df_img12, x=Path(i1).name, hue='Exclosure', ax=ax, fill=True)
sns.kdeplot(data=df_img12, x=Path(i2).name, hue='Exclosure', ax=ax, fill=True)

In [ ]:
# Make a df from each
pix1_overlap_df = pix1_overlap.to_dataframe(name=Path(i1).name)
pix2_overlap_df = pix2_overlap.to_dataframe(name=Path(i2).name)

# Drop na rows
pix1_overlap_df.dropna(axis=0, inplace=True)
pix2_overlap_df.dropna(axis=0, inplace=True)

# Drop cols
pix1_overlap_df.drop(['band', 'spatial_ref'], axis=1, inplace=True)
pix2_overlap_df.drop(['band', 'spatial_ref'], axis=1, inplace=True)

pix12_overlap_df = pd.merge(pix1_overlap_df, pix2_overlap_df, on=['y', 'x'])

In [ ]:
pix12_overlap_xr = pix12_overlap_df.to_xarray()

In [ ]:
pix12_overlap_xr

### 2) KS Test on areas inside exclosures zones, testing whether they record similar temperature values

Added 7/5/22

In [ ]:
# Group df by full and partial
df_pixels_full_g = df_pixels.loc[df_pixels.Exclosure == 'Full'].groupby(['imgf'])
df_pixels_partial_g = df_pixels.loc[df_pixels.Exclosure == 'Partial'].groupby(['imgf'])

# Initilize lists of dfs 
df_full_list = []
df_partial_list = []
img_full_list = []
img_partial_list = []

# For each image
# Takes about 12 seconds per image, so wait ~4 min for 20 images
for i in df_pixels.imgf.unique():
    
    start = time.time()
    
    # Make a temporary img dataframe
    df_img = df_pixels.loc[df_pixels.imgf == i]
    
    # Note: to speed up, maybe add a check here 
    # to see whether this image has fulls, partials, or both, 
    # Then do the filtering based on that (so you don't have to do both each time)
    df_img_full = df_img.loc[df_img.Exclosure == 'Full']
    df_img_partial = df_img.loc[df_img.Exclosure == 'Partial']
    
    # Compare the temperature values for this image to all other images
    # Partials to partials, Fulls to fulls
    if df_img_full.shape[0] > 0:
        
        # Compute KS test 
        df_full_ks = df_pixels_full_g.apply(
            lambda x: ks_2samp(x.Temperature, df_img_full.Temperature).pvalue)
        
        df_full_list.append(df_full_ks)
        img_full_list.append(i)
        
    if df_img_partial.shape[0] > 0:
        
        df_partial_ks = df_pixels_partial_g.apply(
            lambda x: ks_2samp(x.Temperature, df_img_partial.Temperature).pvalue)
        
        df_partial_list.append(df_partial_ks)
        img_partial_list.append(i)
        
    
    end = time.time()
    print(f'{end - start} seconds for image {i}')

# Combine results, and save
df_full_ks_all = pd.concat(df_full_list, axis=1, keys=img_full_list)
df_partial_ks_all = pd.concat(df_partial_list, axis=1, keys=img_partial_list)
#save
df_full_ks_all.to_csv(f'./data/out/{projstr}_KSTest_FullExclosure.csv')
df_partial_ks_all.to_csv(f'./data/out/{projstr}_KSTest_PartialExclosure.csv')
